# Customer Churn Prediction & Recommendation System

## Phase 5 – Feature Engineering

### Objectives

- Prepare data for machine learning
- Encode categorical variables
- Scale numerical features
- Split data into training and testing sets
- Save preprocessing objects

### Import Libraries

In [21]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

### Import Custom Modules

In [22]:
import sys

sys.path.append("..")

from src.preprocessing.data_loader import load_data

### Load Dataset

In [23]:
df = load_data("../data/processed/Telco_Customer_Churn_Cleaned.csv")

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Dataset info

In [24]:
print(df.shape)

df.info()

(7043, 20)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    


In [25]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing values with the median
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Verify
print(df["TotalCharges"].dtype)

float64


### Separate Features and Target

In [26]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [27]:
print(X.shape)

print(y.shape)

(7043, 19)
(7043,)


### Identify Column type 

In [28]:
categorical_columns = X.select_dtypes(include="object").columns.tolist()

numerical_columns = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

print()

print("Numerical Columns:")
print(numerical_columns)

Categorical Columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical Columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\M Ismail\AppData\Local\Temp\ipykernel_3468\909928863.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(include="object").columns.tolist()


### Encode Target Variable

In [29]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print(y[:10])

[0 0 1 0 1 1 0 0 1 0]


## Test-Train Split

In [30]:
from sklearn.model_selection import train_test_split

X = df.drop("Churn", axis=1)
y = df["Churn"]

# Encode target variable
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

Training Features : (5634, 19)
Testing Features  : (1409, 19)
Training Labels   : (5634,)
Testing Labels    : (1409,)


### Create Feature Engineering Pipeline

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Create pipeline

In [32]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine both pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_columns),
        ("cat", categorical_transformer, categorical_columns)
    ]
)

print("✅ Preprocessing pipeline created successfully!")

✅ Preprocessing pipeline created successfully!


### pipeline Apply

In [33]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Data Shape :", X_train_processed.shape)
print("Processed Testing Data Shape  :", X_test_processed.shape)

Processed Training Data Shape : (5634, 45)
Processed Testing Data Shape  : (1409, 45)


### Save Preprocessing Pipeline

In [34]:
import os
import joblib

os.makedirs("../models/preprocessors", exist_ok=True)

joblib.dump(preprocessor, "../models/preprocessors/preprocessing_pipeline.pkl")
joblib.dump(target_encoder, "../models/preprocessors/target_encoder.pkl")

print("✅ Preprocessing pipeline saved successfully!")

✅ Preprocessing pipeline saved successfully!
